# 21. Ders: Yapay Zeka ile Görüntü Sınıflandırma (Ne Görüyorsun?)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Önceki dersimizde, bilgisayara piksellerdeki ışık ve gölgeleri takip ederek bir 'yüz' bulmayı (Face Detection) öğrettik. Ancak yapay zekanın sadece o kutunun nerede olduğunu bulması yetmez. Otonom bir araç (Togg, Tesla) sürüş halindeyken kamerasına yansıyan şeyin bir 'Yaya' mı, 'Trafik Levhası' mı yoksa 'Plastik Kutu' mu olduğunu bilmek zorundadır (Çünkü yayaya çarpamazsınız, ama kutuyu ezebilirsiniz).

Bilgisayarın kendisine gösterilen görüntünün NE OLDUĞUNU tahmin etmesine 'Görüntü Sınıflandırma' (Image Classification) denir. Bu dersimizde Hugging Face'te bulunan, milyonlarca kedi, köpek, araba, ev fotoğrafıyla daha önceden eğitilmiş devasa bir derin öğrenme (Deep Learning) modelini kullanarak resimdeki nesneleri tanıyacağız.

## Adım 2: Teknik Terimler ve Sözlük

- Image Classification (Görüntü Sınıflandırma): Verilen bütün bir resmin ana temasını (Örn: "Bu resimde bir Köpek var") etiketleme işidir.
- CNN (Convolutional Neural Networks - Evrişimli Sinir Ağları): Görüntü işlemek için icat edilmiş, görseldeki kenarları, kıvrımları ve dokuları filtreleyen ünlü yapay zeka algoritması.
- ViT (Vision Transformer): CNN'lerin tahtını sallayan, görüntüyü bir kelime (cümle) gibi okuyup yorumlayan en modern yeni nesil yapay zeka mimarisidir.
- Label (Etiket): Modelin resimde tespit ettiği nesnenin İngilizce adıdır.

## Adım 3: Hugging Face Sınıflandırma Pipeline'ını Hazırlama

Doğal dil işlemede (NLP) kullandığımız `pipeline` aracını bu kez bilgisayarlı görme (Computer Vision) görevi olan `image-classification` için kullanacağız.

In [ ]:
# Hugging Face Transformers paketinden pipeline çağırıyoruz
from transformers import pipeline
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

# Varsayılan model olarak dünyaca ünlü ViT (Vision Transformer) modelini indirir. 
# (Bu model 1000'den fazla farklı nesneyi tanıyacak şekilde ImageNet üzerinde eğitilmiştir)
# Not: İndirme işlemi biraz sürebilir (Ortalama 350 MB).
gorus_modeli = pipeline("image-classification")

print("Görüntü Sınıflandırma (ViT) Yapay Zekası görmeye hazır!")

## Adım 4: Test İçin İnternetten Bir Fotoğraf İndirme (Panda)

Önce yapay zekanın ne kadar başarılı olduğunu görmek için bir panda fotoğrafı indirelim ve ekrana çizdirelim.

In [ ]:
from PIL import Image
import requests

# Bir panda fotoğrafı URL'si
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/0/0f/Grosser_Panda.JPG/800px-Grosser_Panda.JPG"

# Görüntüyü internetten RAM'e çekip açıyoruz (OpenCV yerine PIL kullanıyoruz çünkü HF modelleri PIL sever)
resim = Image.open(requests.get(url, stream=True).raw)

# Resmi colab'de gösterelim
display(resim.resize((400, 300)))

## Adım 5: Fotoğrafı Yapay Zekaya Verip Analiz Ettirme

Hafızadaki resmi `gorus_modeli`nin içine atalım ve bize ne göreceğini söylemesini bekleyelim.

In [ ]:
# Analiz başlatılıyor...
tahminler = gorus_modeli(resim)

print("Yapay Zekanın Tahmin Çıktısı (Ham Veri):")
print(tahminler)

# Çıktıda bir liste içinde 5 farklı olası cevap ve onların skorları (eminlik oranları) gelir.

## Adım 6: Çıktıdaki Etiketleri (Label) Okuma ve Filtreleme

Sistem genellikle en çok benzeyen 5 nesneyi sıralar. Biz en üstteki (0. sıradaki) en güçlü tahmini çekip, İngilizce etiketini (label) kullanıcıya göstereceğiz.

In [ ]:
# En yüksek ihtimalli ilk tahmini al
en_iyi_tahmin = tahminler[0]

etiket = en_iyi_tahmin['label']
skor = en_iyi_tahmin['score']

print("--- YAPAY ZEKA GÖRÜŞ RAPORU ---")
# Büyük harfe çevirip (.upper) ihtimalini yüzde olarak yazdıralım
print(f"TESPİT EDİLEN NESNE: {etiket.upper()}")
print(f"EMİNLİK ORANI (GÜVEN): % {skor * 100:.2f}")

# Modelin "giant panda, panda, panda bear, coon bear" sonucunu 
# %99'un üzerinde bir eminlikle bulduğunu göreceksiniz.

## Adım 7: İkinci ve Üçüncü Tahminlerin İncelenmesi

Peki ya model yanılıyorsa? Veya fotoğrafta birden fazla bağlam varsa? Bir for döngüsü ile diğer olasılıklara da (Alternatif Evrenlere) göz atalım.

In [ ]:
print("Modelin Diğer Alternatif Düşünceleri:")
# 1. indeksten (yani ikinci tahminden) sonuna kadar dön
for i, tahmin in enumerate(tahminler[1:]):
    # i normalde 0'dan başlar, biz onu 2. Tahmin yapmak için i+2 dedik
    print(f"{i+2}. İhtimal: {tahmin['label']} (Skor: % {tahmin['score']*100:.2f})")

# Göreceğiniz gibi diğer ihtimaller komik derecede düşüktür (Örn: %0.15 ihtimalle oyuncak ayı - teddy bear)

## Adım 8: Dene ve Keşfet - Farklı Objelerle Modeli Zorlamak

Adım 4'teki URL değişkenine gidip, internetten bulduğunuz bir araba, uçak, pizza veya kahve fincanı fotoğrafının linkini yapıştırın. Adım 4'ten itibaren kodları sırayla çalıştırıp yapay zekanın nesneleri doğru bilip bilemediğini kontrol edin.

*Not: Model sadece üzerine eğitildiği 1000 genel kategoriyi (ImageNet veri seti) bilir. Eğer çok fantastik veya yeni çıkmış bir ürün resmi verirseniz model en çok benzettiği başka bir şeye uydurma (halüsinasyon) eğiliminde olacaktır.*

## Adım 9: Sıra Sende Görevi - Otomatik Hashtag Üretici Bot

Problem: Bir Instagram benzeri fotoğraf uygulamanız var. Kullanıcılar fotoğraf yüklediğinde, altına yazacak hashtag'leri (etiketleri) bulmakta zorlanıyorlar. Fotoğrafa bakıp otomatik olarak `#` işaretiyle 3 adet hashtag öneren bir fonksiyon yazmanız isteniyor.

Gereksinimler:
1. `otomatik_hashtag_uret(resim_url)` adında bir fonksiyon (def) yazın.
2. İçerisinde requests ve Image (PIL) kullanarak resmi URL'den indirip açın.
3. Resmi `gorus_modeli` içine gönderip analiz sonuçlarını alın.
4. Sadece ilk 3 tahminin `label` değerini çekin. 
5. Bazen labellar virgüllü olabilir (Örn: `giant panda, panda`). `.split(',')[0]` kullanarak sadece ilk kelimesini alın.
6. Çıkan kelimenin başındaki boşlukları silip başına `#` işareti ekleyerek ekrana yazdırın. (Örn: `#giantpanda`)
7. Fonksiyonunuzu istediğiniz bir fotoğraf linkiyle test edin.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 21. Dersi tamamlayarak yapay zekanın sadece metinleri (NLP) değil, aynı zamanda fotoğrafları (CV) da bir insan gibi tanıyabildiğini kanıtladınız. Bu model altyapısı Tesla'nın otomatik pilotunda, otonom drone'larda ve Google Lens gibi uygulamalarda kullanılmaktadır.

Bir sonraki dersimizde, yapay zekanın dünyadaki en korkutucu ama bir o kadar da havalı yeteneklerinden biri olan 'Görüntü Üretimi' (Image Generation - Text to Image) konusuna geçiş yapacağız ve kendi resimlerimizi çizdireceğiz.